# PFCA cross-audit: big models on Kaggle

This notebook runs the 21 large-model evaluations that do not fit a 6 GB laptop GPU, in **7 sessions (A to G)**.
Each session stays under Kaggle's 12-hour limit.

**Every session:**
1. Settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On**.
2. In the cell below, set `SESSION` to the next letter.
3. Click **Save Version**, then choose **Save & Run All (Commit)**. It runs in the background; you can close the browser.
4. When it finishes, open the version's **Output** tab and download `results_session_<X>.zip`.
   Unzip each one into the `results/` folder of this repository; them into `results/`.

| Session | What runs | Approx. GPU time |
|---|---|---|
| A | ColPali + ColQwen2 (A3 IRPAPERS), LLAMA-7B (A2 x 4 datasets), Instructor-XL (A1 LitSearch, MAPLE) | ~7 h |
| B | GritLM-7B: A1 LitSearch, MAPLE, IRPAPERS | ~7 h |
| C | Instructor-XL: A1 and A4 DORIS-MAE | ~5.5 h |
| D | Instructor-XL A4 LitSearch; Qwen3-Embed-8B A4 LitSearch + IRPAPERS (512 tokens) | ~7.5 h |
| E | GritLM-7B: A4 LitSearch + IRPAPERS (512 tokens) | ~6.5 h |
| F | Instructor-XL A4 MAPLE; Qwen3-Embed-8B A4 MAPLE (512 tokens) | ~8.5 h |
| G | GritLM-7B: A4 MAPLE (512 tokens) | ~7.5 h |

Finished models are written to `/kaggle/working/results` immediately, so a session that stops early keeps what it finished.
The three 7-8B runs on DORIS-MAE's 363K abstracts are intentionally not run (too large for free GPUs).

In [ ]:
SESSION = "A"   # <-- change to B, C, D, E, F, G for the next sessions


## Setup (copies the kit, installs this session's packages)

In [ ]:
import os, glob, shutil, subprocess, sys, time
assert SESSION in "ABCDEFG" and len(SESSION) == 1, "SESSION must be one letter A-G"
src = os.path.dirname(glob.glob("/kaggle/input/**/xaudit-kit/FRAMEWORK.md", recursive=True)[0])
KIT = "/tmp/kit"
if not os.path.exists(KIT):
    shutil.copytree(src, KIT)
os.chdir(KIT)
os.environ["XAUDIT_RESULTS"] = "/kaggle/working/results"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs("/kaggle/working/results/tasks", exist_ok=True)
print("kit:", src, "->", KIT, "| shipped task files:", sorted(os.listdir("data/cache")))
PIP = {"A": ['colpali-engine>=0.3.12', 'sentence-transformers>=3.1', 'PyStemmer'],
       "B": ['gritlm', 'PyStemmer'], "E": ['gritlm', 'PyStemmer'], "G": ['gritlm', 'PyStemmer'],
       "C": ['sentence-transformers>=3.1', 'PyStemmer'],
       "D": ['sentence-transformers>=3.1', 'transformers>=4.51', 'accelerate', 'PyStemmer'],
       "F": ['sentence-transformers>=3.1', 'transformers>=4.51', 'accelerate', 'PyStemmer']}[SESSION]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *PIP], check=True)
import torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 1, "No GPU: set Accelerator = GPU T4 x2 in the notebook settings."


## Data this session needs

In [ ]:
NEED = {"A": ["irpapers", "litsearch", "maple"], "B": ["litsearch", "maple"], "C": ["doris_mae"], "D": ["litsearch"], "E": ["litsearch"],
        "F": ["maple"], "G": ["maple"]}[SESSION]
if NEED:
    t = time.time()
    subprocess.run([sys.executable, "download_data.py", *NEED], check=True)
    print(f"downloaded {NEED} in {(time.time() - t) / 60:.1f} min")


## Runs (each finished model is saved immediately)

In [ ]:
# (arch, dataset, model, batch size, max tokens or None)
PLAN = {
 "A": [("A3", "irpapers", "colpali", 2, None), ("A3", "irpapers", "colqwen2", 2, None),
       ("A2", "doris_mae", "llama-7b", 8, None), ("A2", "irpapers", "llama-7b", 8, None),
       ("A2", "litsearch", "llama-7b", 8, None), ("A2", "maple", "llama-7b", 8, None),
       ("A1", "litsearch", "instructor-xl-litsearch", 32, None), ("A1", "maple", "instructor-xl-litsearch", 32, None)],
 "B": [("A1", "irpapers", "gritlm-7b-litsearch", 8, None), ("A1", "litsearch", "gritlm-7b-litsearch", 8, None),
       ("A1", "maple", "gritlm-7b-litsearch", 8, None)],
 "C": [("A1", "doris_mae", "instructor-xl-litsearch", 32, None), ("A4", "doris_mae", "instructor-xl-maple", 32, None)],
 "D": [("A4", "irpapers", "qwen3-embed-8b", 4, 512), ("A4", "litsearch", "qwen3-embed-8b", 4, 512),
       ("A4", "litsearch", "instructor-xl-maple", 32, None)],
 "E": [("A4", "irpapers", "gritlm-7b-maple", 4, 512), ("A4", "litsearch", "gritlm-7b-maple", 4, 512)],
 "F": [("A4", "maple", "qwen3-embed-8b", 4, 512), ("A4", "maple", "instructor-xl-maple", 32, None)],
 "G": [("A4", "maple", "gritlm-7b-maple", 4, 512)],
}[SESSION]
log = f"/kaggle/working/session_{SESSION}_log.txt"
for arch, ds, model, bs, ml in PLAN:
    cmd = [sys.executable, "-u", "-m", "xaudit.run", "--arch", arch, "--dataset", ds, "--model", model,
           "--batch-size", str(bs)] + (["--max-len", str(ml)] if ml else [])
    print(time.strftime("%H:%M"), " ".join(cmd[3:]), flush=True)
    with open(log, "a") as f:
        f.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} {' '.join(cmd[3:])}\n"); f.flush()
        subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
    for line in open(log).read().splitlines()[-3:]:
        print("   ", line[-200:])


## Pack the results for download

In [ ]:
import json, zipfile
out = f"/kaggle/working/results_session_{SESSION}.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob("/kaggle/working/results/*.json") + glob.glob("/kaggle/working/results/tasks/*.json"):
        z.write(p, os.path.relpath(p, "/kaggle/working"))
    z.write(log, os.path.basename(log))
for p in sorted(glob.glob("/kaggle/working/results/*__*__*.json")):
    r = json.load(open(p)); print(f"{r['arch']} {r['dataset']:10s} {r['model_key']:24s} {r['status']:7s} {r.get('recall@10')}  {r.get('reason', '')[:120]}")
print("Download:", out)
